# Tarea 4. Reporte de práctica de red neuronal ADALINE para regresión.

**Integrantes:**

- Valeria Esmer Lobo

- Dayanni Godoy Rosales

- Estephany Lisset Herrera Placencia

- Maria Fernanda Vargas Vaca

**Fecha:**
16 de Octubre de 2026 

**Institución:**
Instituto Tecnológico y de Estudios Superiores de Occidente

**Materia:** Métodos de Inteligencia Artificial

## Desarrollo

### Definición del problema
El problema a resolver consiste en la **predicción de series temporales financieras**, específicamente el precio de cierre de un activo en el mercado de valores (en este caso, acciones de Google), utilizando un enfoque basado en **rezagos temporales (lags)**. 

En los mercados financieros, se asume que el comportamiento histórico reciente de un activo contiene información valiosa sobre su tendencia a corto plazo. Por lo tanto, el problema se modela matemáticamente tomando el precio de un día específico como la variable dependiente (etiqueta a predecir) y los precios de los $N$ días anteriores como las variables independientes (características o rezagos). De esta manera, transformamos una secuencia lineal de tiempo en un problema de aprendizaje supervisado de regresión, donde un modelo lineal (como la red ADALINE) buscará encontrar los pesos óptimos que relacionen los precios pasados con el precio futuro, permitiendo así pronosticar el valor del activo en el día de mañana.

### Descripción del algoritmo

La **Red Neuronal ADALINE (ADAptive LInear NEuron)** es un algoritmo de aprendizaje automático supervisado, desarrollado inicialmente para tareas de clasificación binaria, pero que, gracias a su arquitectura, es ampliamente utilizado para problemas de **regresión lineal**. 

A diferencia del Perceptrón clásico, que actualiza sus pesos basándose en una función de activación escalón (salida binaria), ADALINE utiliza una **función de activación lineal**. Esto significa que la salida de la red es una combinación lineal directa de las entradas y sus respectivos pesos sinápticos ($Y = X \cdot W$). 

Para el proceso de aprendizaje, ADALINE emplea el **descenso de gradiente** a través de la **Regla de Widrow-Hoff** (también conocida como regla delta). El objetivo de esta regla es minimizar una función de costo, típicamente el Error Cuadrático Medio (MSE), calculando la diferencia continua entre la salida real y la salida lineal estimada por la red, antes de pasar por cualquier cuantificador. Al ajustar los pesos en dirección contraria al gradiente del error, el modelo converge hacia los pesos óptimos. 

*Nota sobre la implementación:* Aunque el algoritmo ADALINE estándar encuentra los pesos de forma iterativa, el problema subyacente que resuelve es de naturaleza lineal. Por ello, matemáticamente es equivalente a resolver el sistema utilizando la ecuación normal de **Mínimos Cuadrados** ($W = (X^T X)^{-1} X^T Y$), método analítico directo que se refleja en nuestro script de implementación para encontrar los pesos óptimos en un solo paso.

### Pseudocódigo de ADALINE 

El proceso iterativo del algoritmo ADALINE consta de los siguientes 6 pasos fundamentales:

INICIO
    Descargar historial de precios del activo en el rango de fechas definido
    Extraer y almacenar los precios de cierre en el vector 'precios'
    Definir el número de rezagos nrez = 10
    Inicializar lista X vacía

    PARA k = 0 HASTA nrez HACER
        Extraer subconjunto de 'precios' desplazado k posiciones
        Agregar el subconjunto como columna a X
    FIN PARA

    Definir vector de etiquetas Y como la primera columna de X (precio actual)
    Construir matriz Xa uniendo una columna de unos (sesgo) y las columnas restantes de X (rezagos)
    
    Calcular ntrain como el 70% de la cantidad total de datos
    Separar Xa y Y en conjuntos de entrenamiento (Train) y prueba (Test) usando ntrain

    Calcular pesos Wmc mediante mínimos cuadrados: Wmc = (Xa_trainᵀ * Xa_train)⁻¹ * Xa_trainᵀ * Y_train
    Calcular salida estimada de entrenamiento Yg_mc = Xa_train * Wmc
    Calcular salida estimada de prueba Yg_rec = Xa_test * Wmc

    Inicializar vector 'last' con un 1 en la primera posición
    Asignar a 'last' los últimos 'nrez' precios invertidos en las posiciones restantes
    Calcular predicción del día de mañana = last * Wmc

    Mostrar en consola el último precio real y la predicción del día de mañana
    Graficar los precios reales originales
    Graficar la estimación del modelo en entrenamiento y en prueba
    Graficar el punto de la predicción del día de mañana
FIN

Este es el código para esta tarea:

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def download_values(ticker, start_date, end_date, interval='d', data_type='history'):
    # Mapear los intervalos cortos al formato requerido por yfinance
    interval_map = {
        'd': '1d',  # Diario
        'w': '1wk', # Semanal
        'm': '1mo'  # Mensual
    }
    yf_interval = interval_map.get(interval.lower(), '1d')
    
    # Inicializar el objeto Ticker
    stock = yf.Ticker(ticker)
    
    if data_type.lower() == 'history':
        # Descargar el historial de precios
        data = stock.history(
            start=start_date, 
            end=end_date, 
            interval=yf_interval
        )
        
        # Eliminar columnas de zona horaria si es necesario (opcional)
        if not data.empty:
            data.index = data.index.tz_localize(None)
            
        return data
        
    elif data_type.lower() == 'dividends':
        # Retornar solo dividendos en ese rango de fechas
        dividends = stock.dividends
        if not dividends.empty:
            dividends.index = dividends.index.tz_localize(None)
            return dividends.loc[start_date:end_date]
        return dividends
        
    else:
        raise ValueError("El data_type debe ser 'history' o 'dividends'.")

# Ejecución solicitada
res = download_values('GOOG', '2020-10-06', '2023-10-06', 'd', 'history')

# Mostrar las primeras filas del resultado
print("Datos descargados:")
print(res.head())

# Almacenar los datos
precios = res["Close"].to_numpy()

# Número de rezagos en el tiempo
nrez = 50

X = []

for k in range(nrez + 1):
    X.append(precios[nrez - k: len(precios) - k])

X = np.column_stack(X)
m_filas = X.shape[0]

# Datos de entrenamiento Y y etiqueta Xa = x asterisco [1,x1,x2]
Y = X[:,0].reshape(-1,1)
Xa = np.column_stack((np.ones(m_filas), X[:,1:]))

# Partición de los datos de entrenamiento (70% Train y 30% Test)
ntrain = int(np.round(0.7 * Xa.shape[0]))

Xa_train = Xa[:ntrain, :]
Xa_test = Xa[ntrain:,:]
Y_train = Y[:ntrain,:]
Y_test = Y[ntrain:,:]

# Mínimos cuadrados
Wmc = np.linalg.inv(Xa_train.T @ Xa_train) @ Xa_train.T @ Y_train

# Y estimada
Yg_mc = Xa_train @ Wmc
Yg_rec = Xa_test @ Wmc

# ==========================================
# PREDICCIÓN PARA M DÍAS (NUEVO BLOQUE)
# ==========================================
m_dias = 10 # <--- Define aquí cuántos días al futuro quieres predecir
predicciones_futuras = []

# Tomamos los últimos 'nrez' precios reales y los invertimos para el formato de rezago
current_lags = precios[-nrez:][::-1]

for i in range(m_dias):
    # Preparar el vector de entrada con el 1 inicial (bias/sesgo)
    last = np.zeros((1, nrez + 1))
    last[0, 0] = 1
    last[0, 1:] = current_lags
    
    # Predecir el siguiente día
    new_price = last @ Wmc
    prediccion_actual = new_price[0, 0]
    predicciones_futuras.append(prediccion_actual)
    
    # Actualizar los rezagos para la siguiente iteración:
    # Insertamos la nueva predicción al inicio y eliminamos el último valor
    current_lags = np.insert(current_lags, 0, prediccion_actual)[:-1]

print(f"\nPrecio actual (último día real): {Y[-1,0]:.4f}")
print(f"Predicciones para los próximos {m_dias} días:")
for i, p in enumerate(predicciones_futuras):
    print(f"  Día {i+1}: {p:.4f}")

# ==========================================
# GRÁFICA ACTUALIZADA
# ==========================================
plt.figure(figsize=(12, 6))

# Graficar datos reales e históricos ajustados
plt.plot(range(len(Y)), Y, "b-", linewidth=2, label="Precios Reales")
plt.plot(range(len(Yg_mc)), Yg_mc, "r", linewidth=2, label="Entrenamiento (Train)")
plt.plot(range(ntrain, ntrain + len(Yg_rec)), Yg_rec, "g", linewidth=2, label="Prueba (Test)")

# Graficar la curva de predicción de m días hacia el futuro
rango_futuro = range(len(Y), len(Y) + m_dias)
plt.plot(rango_futuro, predicciones_futuras, "m*-", markersize=10, linewidth=2, label=f"Predicción ({m_dias} días)")

plt.grid(True)
plt.legend(loc="best", fontsize=12)
plt.title(f"Modelo AR({nrez}) - Predicción a {m_dias} días para GOOG")
plt.tight_layout()
plt.show()